In [ ]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

URL = "https://raw.githubusercontent.com/chuksoo/IBM-Data-Science-Capstone-SpaceX/main/spacex_launch_geo.csv"
spacex_df = pd.read_csv(URL)

spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]

nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for _, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    site_map.add_child(
        folium.Circle(
            coordinate,
            radius=1000,
            color='#d35400',
            fill=True
        ).add_child(folium.Popup(row['Launch Site']))
    )
    site_map.add_child(
        folium.Marker(
            coordinate,
            icon=DivIcon(
                icon_size=(20, 20),
                icon_anchor=(0, 0),
                html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site']
            )
        )
    )

marker_cluster = MarkerCluster()
spacex_df['marker_color'] = spacex_df['class'].map({1: 'green', 0: 'red'})
site_map.add_child(marker_cluster)

for _, record in spacex_df.iterrows():
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color'])
    )
    marker_cluster.add_child(marker)

formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter
)
site_map.add_child(mouse_position)

ccafs = launch_sites_df[launch_sites_df['Launch Site'] == 'CCAFS LC-40'].iloc[0]
launch_site_lat = ccafs['Lat']
launch_site_lon = ccafs['Long']

coastline_lat, coastline_lon = 28.56367, -80.57163
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

site_map.add_child(
    folium.Marker(
        [coastline_lat, coastline_lon],
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline)
        )
    )
)

site_map.add_child(
    folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]],
        weight=1
    )
)

city_lat, city_lon = 28.40192, -80.60433
distance_city = calculate_distance(launch_site_lat, launch_site_lon, city_lat, city_lon)

site_map.add_child(
    folium.Marker(
        [city_lat, city_lon],
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#2c3e50;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_city)
        )
    )
)

site_map.add_child(
    folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], [city_lat, city_lon]],
        weight=1,
        color='gray'
    )
)

railway_lat, railway_lon = 28.57210, -80.58527
distance_railway = calculate_distance(launch_site_lat, launch_site_lon, railway_lat, railway_lon)

site_map.add_child(
    folium.Marker(
        [railway_lat, railway_lon],
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#7f8c8d;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_railway)
        )
    )
)

site_map.add_child(
    folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], [railway_lat, railway_lon]],
        weight=1,
        color='blue'
    )
)

highway_lat, highway_lon = 28.56320, -80.57080
distance_highway = calculate_distance(launch_site_lat, launch_site_lon, highway_lat, highway_lon)

site_map.add_child(
    folium.Marker(
        [highway_lat, highway_lon],
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#8e44ad;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_highway)
        )
    )
)

site_map.add_child(
    folium.PolyLine(
        locations=[[launch_site_lat, launch_site_lon], [highway_lat, highway_lon]],
        weight=1,
        color='purple'
    )
)

site_map